<a href="https://colab.research.google.com/github/sardorbekmuminov/attention-from-scratch/blob/main/text_summarization_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Matn xulosasi (Summarization) — BART bilan

**Maqsad:** uzun inglizcha matn beramiz, model uning qisqa xulosasini yozib beradi.

**Dars 67 bilan farqi:**

| | Sentiment (Dars 67) | Xulosa (bu daftar) |
|---|---|---|
| Natija | `POSITIVE` / `NEGATIVE` | **yangi matn** |
| Model sinfi | `AutoModelForSequenceClassification` | `AutoModelForSeq2SeqLM` |
| Javob olish | `model(**batch)` → logits | `model.generate(**batch)` → token ID lar |

**Bo'limlar:**
1. Tayyorgarlik
2. Model va tokenizer
3. Matn va tokenlash
4. Xulosa chiqarish (`generate`)
5. Funksiya va parametrlar
6. Uzun matn
7. Topshiriqlar
8. Xulosa va cheklovlar

> Boshlashdan oldin: `Runtime → Change runtime type → T4 GPU`.

## 1-BO'LIM: Tayyorgarlik

In [1]:
!pip install -q -U transformers
import torch, transformers
print("transformers:", transformers.__version__)
print("GPU bor:", torch.cuda.is_available())

transformers: 5.23.0
GPU bor: True


- Qizil `ERROR ... pyarrow / cudf` yozuvi chiqsa, e'tibor bermang: bu Colab dagi boshqa paket haqida ogohlantirish.
- `pipeline("summarization")` **transformers 5 da yo'q** (olib tashlangan). Shuning uchun biz modelni qo'lda ishlatamiz: tokenizer + model + `generate`. Bu ichida nima bo'layotganini tushunish uchun ham foydali.

## 2-BO'LIM: Model va tokenizer

`facebook/bart-large-cnn` — yangiliklarni xulosa qilishga o'qitilgan model.

**Encoder-decoder** nima? Model ikki qismdan iborat:
- **Encoder** matnni o'qiydi va tushunadi,
- **Decoder** xulosani so'zma-so'z yozadi.

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL = "facebook/bart-large-cnn"

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)   # modelni GPU ga ko'chiramiz
model.eval()               # javob berish rejimi (o'qitish emas)

print("Parametrlar soni:", sum(p.numel() for p in model.parameters()))
print("Kirish chegarasi (token):", model.config.max_position_embeddings)

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Parametrlar soni: 406290432
Kirish chegarasi (token): 1024


Nimani ko'rdik:
- `from_pretrained(MODEL)` ni tokenizer va model uchun **bir xil nomdan** yukladik (Dars 67 qoidasi).
- `model.to(device)` — model va keyin kiritiladigan ma'lumot **bir xil joyda** (GPU yoki CPU) bo'lishi kerak.
- Kirish chegarasi **1024 token**. Undan uzun matn kesiladi. 6-bo'limda buni hal qilamiz.

## 3-BO'LIM: Matn va tokenlash

Test uchun matn. Keyin o'zingizning matningizni qo'yasiz.

In [3]:
text = """A city council in a mid-sized coastal town has approved a plan to replace its aging bus fleet with electric vehicles over the next six years. The decision, passed on Tuesday by a vote of seven to two, follows a pilot programme in which twelve electric buses ran on the busiest routes for eighteen months. Officials said the pilot cut fuel and maintenance costs by almost a third compared with diesel buses on the same lines. Drivers also reported that the vehicles were quieter and easier to handle in heavy traffic.

Under the plan, the first forty buses will be delivered next year, with the rest arriving in annual batches. The total cost is estimated at 120 million, of which about half will come from a national clean transport fund. The remainder will be financed through a municipal bond and savings from lower fuel use. Two council members who voted against the plan argued that the city should wait for battery prices to fall further and warned that charging depots could overload the local power grid.

City engineers responded that new depots will be built with their own solar panels and large batteries, which should reduce demand on the grid at peak hours. They also said that battery warranties of eight years cover the most expensive component of each bus. Passenger groups welcomed the decision but asked the council to protect fares and to keep older buses on rural routes until charging points are available there. The council promised to publish a yearly progress report and to review the schedule after the first two years."""

print(len(text.split()), "ta so'z")

265 ta so'z


In [4]:
batch = tok(text, truncation=True, max_length=1024, return_tensors="pt").to(device)

print(batch["input_ids"].shape)                              # (1, token soni)
print(tok.convert_ids_to_tokens(batch["input_ids"][0][:12])) # birinchi 12 token

torch.Size([1, 293])
['<s>', 'A', 'Ġcity', 'Ġcouncil', 'Ġin', 'Ġa', 'Ġmid', '-', 'sized', 'Ġcoastal', 'Ġtown', 'Ġhas']


- `shape` = `(1, T)`: 1 ta matn, T ta token. Agar T = 1024 bo'lsa, matn **kesilgan**.
- Tokenlar oldida `Ġ` belgisi bor: bu "oldida bo'shliq bor" degani. BART **BPE** usulidan foydalanadi (DistilBERT dagi `##` WordPiece dan farq qiladi, lekin g'oya bir xil: noma'lum so'z bo'laklarga bo'linadi).
- `<s>` va `</s>` — boshlanish va tugash tokenlari (DistilBERT dagi `[CLS]` / `[SEP]` ga o'xshash).

## 4-BO'LIM: Xulosa chiqarish

Sentiment da `model(**batch)` bir marta hisoblab logits berar edi. Bu yerda `model.generate(**batch)` **so'zma-so'z yozadi**: har bir qadamda keyingi tokenni tanlaydi.

In [5]:
with torch.no_grad():
    ids = model.generate(
        **batch,
        max_new_tokens=130,      # xulosa eng ko'pi bilan 130 token
        min_new_tokens=30,       # kamida 30 token
        num_beams=4,             # 4 ta variantni solishtiradi
        no_repeat_ngram_size=3,  # bir xil 3 so'zlik iborani takrorlamaydi
        length_penalty=2.0,
        early_stopping=True,
    )

print(ids.shape)                                   # (1, xulosa uzunligi)
summary = tok.decode(ids[0], skip_special_tokens=True)
print(summary)

[transformers] Both `max_new_tokens` (=130) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=30) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


torch.Size([1, 70])
A city council in a mid-sized coastal town has approved a plan to replace its aging bus fleet with electric vehicles over the next six years. The decision follows a pilot programme in which twelve electric buses ran on the busiest routes for eighteen months. Officials said the pilot cut fuel and maintenance costs by almost a third compared with diesel buses.


Qadamlar:
1. `generate` token **ID lar** qaytaradi (raqamlar).
2. `tok.decode(...)` ularni matnga aylantiradi (Dars 67, 2-bo'lim).
3. `ids[0]` — birinchi (yagona) matn uchun xulosa.
4. `skip_special_tokens=True` — `<s>`, `</s>` kabi maxsus tokenlarni tashlab yuboradi.

Chiqadigan sariq ogohlantirishlar (`max_length`, `forced_bos_token_id`) zararsiz: bizning `max_new_tokens` ustun turadi.

## 5-BO'LIM: Funksiya va parametrlar

Har safar shuncha kod yozmaslik uchun hammasini bitta funksiyaga yig'amiz.

In [6]:
def xulosa(text, max_new_tokens=130, min_new_tokens=30, num_beams=4):
    batch = tok(text, truncation=True, max_length=1024, return_tensors="pt").to(device)
    with torch.no_grad():
        ids = model.generate(
            **batch,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            num_beams=num_beams,
            no_repeat_ngram_size=3,
            length_penalty=2.0,
            early_stopping=True,
        )
    return tok.decode(ids[0], skip_special_tokens=True)

print(xulosa(text))

[transformers] Both `max_new_tokens` (=130) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=30) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A city council in a mid-sized coastal town has approved a plan to replace its aging bus fleet with electric vehicles over the next six years. The decision follows a pilot programme in which twelve electric buses ran on the busiest routes for eighteen months. Officials said the pilot cut fuel and maintenance costs by almost a third compared with diesel buses.


### Parametrlar bilan tajriba
Bir xil matn, faqat bitta parametr o'zgaradi. Farqini o'zingiz kuzating:

In [7]:
import time

for beams in [1, 4]:
    t = time.time()
    s = xulosa(text, num_beams=beams)
    print(f"--- num_beams={beams} ({time.time()-t:.1f} s) ---")
    print(s, "\n")

[transformers] The following generation flags are not valid and may be ignored: ['early_stopping', 'length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=130) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=30) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=130) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` 

--- num_beams=1 (1.8 s) ---
The first forty buses will be delivered next year. The total cost is estimated at 120 million. About half will come from a national clean transport fund. 

--- num_beams=4 (4.1 s) ---
A city council in a mid-sized coastal town has approved a plan to replace its aging bus fleet with electric vehicles over the next six years. The decision follows a pilot programme in which twelve electric buses ran on the busiest routes for eighteen months. Officials said the pilot cut fuel and maintenance costs by almost a third compared with diesel buses. 



In [8]:
for mx in [40, 130]:
    s = xulosa(text, max_new_tokens=mx, min_new_tokens=10)
    print(f"--- max_new_tokens={mx} ({len(s.split())} so'z) ---")
    print(s, "\n")

[transformers] Both `max_new_tokens` (=40) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=10) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=130) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=10) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/d

--- max_new_tokens=40 (34 so'z) ---
A city council in a mid-sized coastal town has approved a plan to replace its aging bus fleet with electric vehicles over the next six years. The decision, passed on Tuesday by a vote 

--- max_new_tokens=130 (62 so'z) ---
A city council in a mid-sized coastal town has approved a plan to replace its aging bus fleet with electric vehicles over the next six years. The decision follows a pilot programme in which twelve electric buses ran on the busiest routes for eighteen months. Officials said the pilot cut fuel and maintenance costs by almost a third compared with diesel buses. 



| Parametr | Ma'nosi |
|---|---|
| `max_new_tokens` | xulosaning maksimal uzunligi |
| `min_new_tokens` | minimal uzunligi |
| `num_beams` | nechta variant solishtiriladi (ko'p = sifatliroq, sekinroq) |
| `no_repeat_ngram_size=3` | takroriy iboralarni to'xtatadi |

## 6-BO'LIM: Uzun matn

Model faqat **1024 token** o'qiydi. Uzun matnning oxiri kesilib, xulosaga kirmay qoladi. Yechim:

```
uzun matn → gaplarga bo'lish → bo'laklarga yig'ish (≤ 900 token)
          → har bir bo'lakdan xulosa → xulosalarni birlashtirib yana xulosa
```

In [9]:
import re

def gaplarga_bol(text):
    text = re.sub(r"\s+", " ", text).strip()
    return re.split(r"(?<=[.!?])\s+", text)

def bolaklarga_bol(text, max_tokens=900):
    bolaklar, hozirgi, hajm = [], [], 0
    for gap in gaplarga_bol(text):
        n = len(tok(gap, add_special_tokens=False)["input_ids"])
        if hozirgi and hajm + n > max_tokens:      # bo'lak to'ldi
            bolaklar.append(" ".join(hozirgi))
            hozirgi, hajm = [], 0
        hozirgi.append(gap)
        hajm += n
    if hozirgi:
        bolaklar.append(" ".join(hozirgi))
    return bolaklar

def uzun_xulosa(text):
    n = len(tok(text, add_special_tokens=False)["input_ids"])
    if n <= 1000:                                   # qisqa: bitta o'tish
        return xulosa(text)
    qismlar = [xulosa(b) for b in bolaklarga_bol(text)]
    birlashgan = " ".join(qismlar)
    return xulosa(birlashgan)                       # xulosalardan xulosa

Sinash uchun uzun matn kerak. Quyida test matnini 8 marta takrorlab uzun qildik (faqat mexanizmni sinash uchun). **Yaxshisi:** Vikipediyadan 1000+ so'zlik inglizcha maqola oling va `uzun_matn` ga qo'ying.

In [10]:
uzun_matn = (text + " ") * 8

print("So'z:", len(uzun_matn.split()))
print("Token:", len(tok(uzun_matn, add_special_tokens=False)["input_ids"]))
print("Bo'laklar soni:", len(bolaklarga_bol(uzun_matn)))
print()
print(uzun_xulosa(uzun_matn))

So'z: 2120
Token: 2329
Bo'laklar soni: 3



[transformers] Both `max_new_tokens` (=130) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=30) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=130) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=30) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

The first forty buses will be delivered next year, with the rest arriving in annual batches. The total cost is estimated at 120 million, of which about half will come from a national clean transport fund.


## 7-BO'LIM: Topshiriqlar

**1 (oson).** Yangi inglizcha yangilik matnini oling (BBC, Guardian yoki Vikipediya) va `xulosa(...)` ga bering. Natija qanday?

**2 (o'rta).** `text` ning dastlabki 2 ta gapini va xulosani solishtiring. Xulosa birinchi gaplardan ko'chirilganmi? Qaysi muhim fakt tushib qolgan?

**3 (o'rta).** `max_new_tokens=60` va `max_new_tokens=200` ni sinab, farqini yozing.

**4 (qiyin).** Boshqa modelni sinang: `MODEL = "sshleifer/distilbart-cnn-12-6"` qilib 2-bo'limdan boshlab qaytadan ishga tushiring. Qaysi biri tezroq? Natijalar qanday farq qildi?

**Bonus:** nega `model.eval()` va `torch.no_grad()` kerak? (Dars 67, 3-bo'limni eslang.)

In [11]:
# 1-topshiriq: o'z matningizni shu yerga qo'ying
oz_matn = """..."""

# print(xulosa(oz_matn))

## 8-BO'LIM: Xulosa va cheklovlar

```
matn → tok() → input_ids → encoder (o'qiydi) → decoder (generate: yozadi) → ID lar → decode → xulosa
```

**Esda tuting:**
- `AutoModelForSeq2SeqLM` + `generate` matn **yozadigan** vazifalar uchun, `AutoModelForSequenceClassification` esa **sinf tanlaydigan** vazifalar uchun.
- Kirish chegarasi 1024 token: uzun matn bo'laklanadi.
- Model yangiliklarda o'qitilgan, shuning uchun **birinchi gaplarni ko'chirishga moyil** ("lead bias"): muhim fakt keyinroq kelsa, tushib qolishi mumkin.
- Model matnda yo'q narsani yozib qo'yishi ham mumkin. Muhim faktlarni asl matn bilan tekshiring.
- Faqat inglizcha matn uchun.

**GitHub ga saqlash:** `File → Save a copy in GitHub`.